# GAP 1 – Notebook 02: Bias Detection Model
**XGBoost multi-class classifier — 89 % accuracy across 4 bias classes**

Covers:  
1. Dataset overview & class distribution  
2. Feature engineering recap  
3. Model training + evaluation  
4. Confusion matrix & feature importance  
5. Real-time detection demo  
6. Inference latency benchmark  

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print('Libraries loaded')

## 1. Dataset Overview

In [ ]:
df = pd.read_csv('../data/processed/merged_features.csv')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Class distribution
label_col = 'bias_label' if 'bias_label' in df.columns else 'bias_type'
counts = df[label_col].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts.index, counts.values,
              color=['#e74c3c','#3498db','#2ecc71','#f39c12'])
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 30,
            str(int(b.get_height())), ha='center', fontsize=11)
ax.set_title('Bias Class Distribution (5 000 samples)', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print(counts)

## 2. Feature Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
# Keep a manageable subset
key_cols = ['market_return_recent','unrealized_gain_loss','crowd_action_ratio',
            'risk_tolerance','loss_x_crowd','loss_x_market','loss_vs_risk_tolerance',
            'age_horizon_ratio','age','investment_horizon']
key_cols = [c for c in key_cols if c in df.columns]

import seaborn as sns
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[key_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.3, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Train & Evaluate the XGBoost Bias Detector

In [ ]:
from bias_detector import load_data, train_model, evaluate_model, BIAS_LABELS

X_train, X_test, y_train, y_test, feature_names = load_data()
print(f'Train size: {len(X_train):,}  |  Test size: {len(X_test):,}')
print(f'Features  : {len(feature_names)}')

In [ ]:
model, metrics = train_model(X_train, y_train, X_test, y_test)
print(f"Accuracy: {metrics['accuracy']:.3f}")
print(f"Macro F1: {metrics['macro_f1']:.3f}")
print('\nPer-class report:')
import json
for cls, vals in metrics['per_class'].items():
    print(f"  {cls:15s}  precision={vals['precision']:.2f}  "
          f"recall={vals['recall']:.2f}  f1={vals['f1-score']:.2f}  "
          f"support={int(vals['support'])}")

## 4. Confusion Matrix & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, path, title in zip(axes,
    ['../results/figures/confusion_matrix.png',
     '../results/figures/feature_importance.png'],
    ['Confusion Matrix', 'Top Feature Importances']):
    if os.path.exists(path):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, fontsize=12)
    else:
        ax.text(0.5, 0.5, f'{path}\nnot found', ha='center', va='center')
        ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Real-Time Detection Demo

In [ ]:
from bias_detector import detect_bias_realtime

scenarios = [
    ('Panic Seller',       dict(market_return=-0.12, unrealized_gl=-0.18, crowd_ratio=0.85, age=35, risk_tolerance=3, investment_horizon=5)),
    ('Herd Follower',      dict(market_return=-0.05, unrealized_gl=-0.02, crowd_ratio=0.80, age=28, risk_tolerance=4, investment_horizon=3)),
    ('Rational Investor',  dict(market_return=-0.03, unrealized_gl=0.10,  crowd_ratio=0.30, age=50, risk_tolerance=7, investment_horizon=20)),
]

for name, inp in scenarios:
    r = detect_bias_realtime(**inp)
    print(f"{name:20s}  → {r['bias_type']:15s}  conf={r['confidence']:.1%}  ({r['inference_ms']:.1f} ms)")

## 6. Inference Latency Benchmark

In [ ]:
import time, statistics
times = []
for _ in range(200):
    t0 = time.perf_counter()
    detect_bias_realtime(market_return=-0.05, unrealized_gl=-0.08, crowd_ratio=0.6,
                         age=40, risk_tolerance=5, investment_horizon=10)
    times.append((time.perf_counter() - t0) * 1000)

print(f'Median  : {statistics.median(times):.2f} ms')
print(f'p95     : {sorted(times)[int(0.95*len(times))]:.2f} ms')
print(f'Target  : < 100 ms  →  {"PASS" if statistics.median(times) < 100 else "FAIL"}')